# Setup e caricamento dei dati

In [1]:
import pandas as pd
import numpy as np
import re
import string
from collections import Counter

# NLTK per la pre-elaborazione del testo
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import LatentDirichletAllocation

# spaCy per NLP avanzato (NER e Word Embeddings)
import spacy
from scipy.spatial.distance import cosine

from sklearn.dummy import DummyClassifier

# caricamento del dataset
try:
    df = pd.read_csv('https://raw.githubusercontent.com/ProfAI/natural-language-processing/refs/heads/main/datasets/Verifica%20Finale%20-%20Spam%20Detection/spam_dataset.csv')
    df = df.drop(columns=['Unnamed: 0'], errors='ignore')
except FileNotFoundError:
    print("Errore: Il file 'spam_dataset.csv' non è stato trovato.")
    exit()

# Esplorazione iniziale dei dati

In [2]:
print(">>>>>>>> Informazioni sul dataset")
print(df.info())
print("\n>>>>>>> Prime 5 righe del dataset")
print(df.head())
print("\n>>>>>> Distribuzione delle etichette")
print(df['label'].value_counts())

>>>>>>>> Informazioni sul dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5171 entries, 0 to 5170
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   label      5171 non-null   object
 1   text       5171 non-null   object
 2   label_num  5171 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 121.3+ KB
None

>>>>>>> Prime 5 righe del dataset
  label                                               text  label_num
0   ham  Subject: enron methanol ; meter # : 988291\nth...          0
1   ham  Subject: hpl nom for january 9 , 2001\n( see a...          0
2   ham  Subject: neon retreat\nho ho ho , we ' re arou...          0
3  spam  Subject: photoshop , windows , office . cheap ...          1
4   ham  Subject: re : indian springs\nthis deal is to ...          0

>>>>>> Distribuzione delle etichette
label
ham     3672
spam    1499
Name: count, dtype: int64


# Pre-elaborazione del testo

In [3]:
# setup robusto NLTK
try:
    #prova a caricare le stopwords
    stop_words = set(stopwords.words('english'))
except LookupError:
    print("Risorsa 'stopwords' di NLTK non trovata. Inizio download...")
    nltk.download('stopwords', quiet=True)
    stop_words = set(stopwords.words('english'))

# verifica della presenza dei tokenizzatori necessari
required_nltk_downloads = ['punkt', 'punkt_tab']
for package in required_nltk_downloads:
    try:
        nltk.data.find(f'tokenizers/{package}')
    except LookupError:
        print(f"Risorsa '{package}' di NLTK non trovata. Inizio download...")
        nltk.download(package, quiet=True)

def preprocess_text(text):
    """
    Funzione per pulire il testo di un'email:
    - Converte in minuscolo
    - Rimuove url, email, numeri e punteggiatura
    - Tokenizza il testo
    - Rimuove le stopwords
    - Ricompone il testo pulito
    """
    if not isinstance(text, str):
        return ""
    # rimuove url, email e caratteri non alfanumerici
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S*@\S*\s?', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))

    # tokenizzazione
    tokens = word_tokenize(text)

    # Rimozione stopwords
    processed_tokens = [word for word in tokens if word not in stop_words and len(word) > 2]

    return " ".join(processed_tokens)

# pre-elaborazione colonna 'text'
print("\n>>>>>>>> Inizio pre-elaborazione del testo")
df['cleaned_text'] = df['text'].apply(preprocess_text)
print(">>>>>>> Pre-elaborazione completata")



>>>>>>>> Inizio pre-elaborazione del testo
>>>>>>> Pre-elaborazione completata


# Addestramento del classificatore spam

In [10]:
print("\n>>>>>>>< Addestramento classificatore spam ---")

#splitting training e test set
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'],
    df['label_num'],
    test_size=0.2,
    random_state=42,
    stratify=df['label_num']
)

# vettorizzazione con TF-IDF
# TF-IDF è efficace per il testo perché pesa le parole in base alla loro importanza nel documento rispetto all'intero corpus
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

#addestramento del modello di regressione logistica: è un modello robusto, veloce e interpretabile, ottimo come baseline
lr_model = LogisticRegression(solver='liblinear', random_state=42)
lr_model.fit(X_train_tfidf, y_train)

# valutazione del modello
y_pred = lr_model.predict(X_test_tfidf)

print("\n>>>>>>>> Risultati del classificatore")
print("\n>>>>>>>>>> Matrice di confusione:")
print(confusion_matrix(y_test, y_pred))

print("\n>>>>> Report di classificazione:")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))



# creazione classificatore che predice sempre la classe più frequente (ham)
dummy_clf = DummyClassifier(strategy='most_frequent')
dummy_clf.fit(X_train_tfidf, y_train)
y_pred_dummy = dummy_clf.predict(X_test_tfidf)

print("\n>>>>>> Report di classificazione (modello baseline)")
print(classification_report(y_test, y_pred_dummy, target_names=['Ham', 'Spam'], zero_division=0))

# --- SIMULAZIONE OPERATIVA: PREDIZIONE SULL'INTERO DATASET ---
print("\n>>>>> Applicazione del modello all'intero dataset")

#trasforma tutto il testo pulito usando il vettorizzatore già addestrato
full_tfidf = tfidf_vectorizer.transform(df['cleaned_text'])

# predizione etichette per ogni email
df['predicted_label'] = lr_model.predict(full_tfidf)

print(f"Numero di email predette come SPAM: {df['predicted_label'].sum()}")
print(f"Numero di email predette come HAM: {len(df) - df['predicted_label'].sum()}")



>>>>>>>< Addestramento classificatore spam ---

>>>>>>>> Risultati del classificatore

>>>>>>>>>> Matrice di confusione:
[[723  12]
 [  8 292]]

>>>>> Report di classificazione:
              precision    recall  f1-score   support

         Ham       0.99      0.98      0.99       735
        Spam       0.96      0.97      0.97       300

    accuracy                           0.98      1035
   macro avg       0.97      0.98      0.98      1035
weighted avg       0.98      0.98      0.98      1035


>>>>>> Report di classificazione (modello baseline)
              precision    recall  f1-score   support

         Ham       0.71      1.00      0.83       735
        Spam       0.00      0.00      0.00       300

    accuracy                           0.71      1035
   macro avg       0.36      0.50      0.42      1035
weighted avg       0.50      0.71      0.59      1035


>>>>> Applicazione del modello all'intero dataset
Numero di email predette come SPAM: 1517
Numero di email predet

# Topic modeling sulle email di spam

In [7]:
print("\n>>>>>>>>>>>>> Topic modeling mail spam")

#filtra solo le email di spam usando l'etichetta reale......Analisi basata sulle etichette predette dal modello
spam_texts_predicted = df[df['predicted_label'] == 1]['cleaned_text']

# vettorizzazione con CountVectorizer per LDA: funziona meglio con conteggi di parole grezze piuttosto che con punteggi TF-IDF.
count_vectorizer = CountVectorizer(max_df=0.95, min_df=2, max_features=1000, stop_words='english')
spam_counts = count_vectorizer.fit_transform(spam_texts_predicted)

#addestramento del modello LDA
#sono stati scelti 5 topic come punto di partenza per l'analisi
n_topics = 5
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42, n_jobs=-1)
lda.fit(spam_counts)

def display_topics(model, feature_names, no_top_words):
    """Funzione per stampare i topic con le parole più rappresentative."""
    topics = {}
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]
        print(f"Topic {topic_idx}: {' | '.join(top_words)}")
        topics[topic_idx] = top_words
    return topics

print("\n>>>>> Principali topic nelle email di spam")
spam_feature_names = count_vectorizer.get_feature_names_out()
spam_topics = display_topics(lda, spam_feature_names, 10)



>>>>>>>>>>>>> Topic modeling mail spam

>>>>> Principali topic nelle email di spam
Topic 0: pills | http | viagra | online | cialis | com | prescription | save | meds | best
Topic 1: com | www | computron | message | contact | free | mail | reply | link | remove
Topic 2: company | font | statements | height | http | information | width | report | size | securities
Topic 3: windows | price | software | adobe | microsoft | professional | office | new | pro | download
Topic 4: nbsp | email | com | http | money | account | time | new | free | like


# Calcolo della distanza semantica tra i topic ed estrazione organizzazioni da mail non spam (ham)

In [9]:
print("\n>>>>>>>>>Calcolo distanza semantica tra topic")

# Caricamento modello spaCy con word embeddings
# 'en_core_web_md' contiene vettori pre-addestrati che permettono di rappresentare parole e documenti in uno spazio semantico
nlp = None
try:
    nlp = spacy.load('en_core_web_md')
    print("Modello spaCy 'en_core_web_md' caricato con successo.")
except OSError:
    print("Modello 'en_core_web_md' non trovato.")
    print("In caso di questo errore eseguire: !python -m spacy download en_core_web_md") ##############In caso di errore allora eseguire questo!

#esecuzione solo se il modello spacy è stato caricato
if nlp:
    def get_topic_vector(topic_words, nlp_model):
        """Calcola il vettore medio per un topic."""
        vectors = [nlp_model(word).vector for word in topic_words if nlp_model(word).has_vector]
        if vectors:
            return np.mean(vectors, axis=0)
        return np.zeros((nlp_model.vocab.vectors_length,))

    #calcolo dei vettori per ogni topic
    topic_vectors = {idx: get_topic_vector(words, nlp) for idx, words in spam_topics.items()}

    #calcolo della matrice di distanza (1 - similarità coseno)
    distance_matrix = pd.DataFrame(
        index=range(n_topics),
        columns=range(n_topics),
        dtype=float
    )

    for i in range(n_topics):
        for j in range(n_topics):
            if i == j:
                distance_matrix.loc[i, j] = 0.0
            else:
                distance_matrix.loc[i, j] = cosine(topic_vectors[i], topic_vectors[j])

    print("\n>>>>>>> Matrice di distanza coseno tra i topic spam") # valori più alti indicano maggiore eterogeneità
    print(distance_matrix)

    # =============ESTRAZIONE ORGANIZZAZIONI DA EMAIL NON SPAM (HAM)==============================================================
    print("\n>>>>> estrazione di organizzazioni dalle email ham")

    #filtra solo le email ham (testo originale per preservare le entità)...Analisi basata sulle etichette predette dal modello
    #viene usato il testo originale text per non perdere le entità
    ham_texts_predicted = df[df['predicted_label'] == 0]['text']

    def extract_organizations(texts, nlp_model):
        """Estrae le entità 'ORG' da una serie di testi."""
        orgs = []
        for doc in nlp_model.pipe(texts, disable=["tok2vec", "parser", "attribute_ruler", "lemmatizer"]):
            for ent in doc.ents:
                if ent.label_ == 'ORG':
                    orgs.append(ent.text.strip())
        return orgs

    # Estrazione delle organizzazioni
    organizations = extract_organizations(ham_texts_predicted, nlp)

    # Conteggio e visualizzazione
    org_counts = Counter(organizations)
    print("\n>>>>> Organizzazioni più menzionate nelle email HAM (top 20)")
    for org, count in org_counts.most_common(20):
        print(f"- {org}: {count} menzioni")


    print("\n>>>>>>>>>>>>> Post elaborazione")
    #lista di falsi-positivi noti da escludere
    blacklist = ['xls', 'doc', 'noms', 'ena', 'tx', 'lauri', 'hanks / hou', 'allen / hou']

    #filtra il contatore
    cleaned_org_counts = Counter()
    for org, count in org_counts.items():
        #rimozione elementi della blacklist, entità troppo corte o che contengono /
        if org.lower() not in blacklist and len(org) > 2 and '/' not in org:
            cleaned_org_counts[org] = count

    print("\n>>>>>>> Organizzazioni più menzionate nelle email ham (top 10)")
    for org, count in cleaned_org_counts.most_common(10):
        print(f"- {org}: {count} menzioni")


>>>>>>>>>Calcolo distanza semantica tra topic
Modello spaCy 'en_core_web_md' caricato con successo.

>>>>>>> Matrice di distanza coseno tra i topic spam
          0         1         2         3         4
0  0.000000  0.495451  0.490555  0.506962  0.366600
1  0.495451  0.000000  0.443432  0.482273  0.280941
2  0.490555  0.443432  0.000000  0.511719  0.391089
3  0.506962  0.482273  0.511719  0.000000  0.373401
4  0.366600  0.280941  0.391089  0.373401  0.000000

>>>>> estrazione di organizzazioni dalle email ham

>>>>> Organizzazioni più menzionate nelle email HAM (top 20)
- enron: 3275 menzioni
- xls: 921 menzioni
- teco: 362 menzioni
- tenaska: 200 menzioni
- ami chokshi / corp / enron: 194 menzioni
- aol: 191 menzioni
- txu: 181 menzioni
- doc: 164 menzioni
- noms: 153 menzioni
- ena: 138 menzioni
- tx: 128 menzioni
- lauri: 112 menzioni
- vance l taylor / hou: 107 menzioni
- hanks / hou: 89 menzioni
- redeliveries: 88 menzioni
- aep: 84 menzioni
- enron north america corp .: 80 men